# DLAV Project - Phase 2

This notebook is the clean Phase 2 entry point for perception-aware planning.

- Core implementation lives in `src/phase2/`.
- Shared run/log/checkpoint helpers still come from `src/shared/`.
- The raw starter notebook remains archived at `notebooks/starter/DLAV_Phase2_starter_reference.ipynb`.


## Edit This Block First

Set the experiment flags in the next cell before running the notebook.


In [ ]:
from pathlib import Path

# ==================================
# Edit this cell before running
# ==================================
# Primary outputs are saved under outputs/runs/phase2/<timestamp>_<run_name>/.

# --- Colab and storage behavior ---
REPO_URL = 'https://github.com/math707/dlav-project.git'
COLAB_PROJECT_DIR = Path('/content/dlav-project')
MOUNT_DRIVE_IN_COLAB = True
COLAB_DRIVE_RUNS_ROOT = Path('/content/drive/MyDrive/dlav-project-runs')
DOWNLOAD_DATA_IF_MISSING = True
SYNC_RUN_TO_DRIVE = True
PHASE_NAME = 'phase2'

# --- Experiment identity and core training settings ---
RUN_NAME = None
MODEL_NAME = 'phase2_multitask'
USE_DEPTH_AUX = True
LAMBDA_DEPTH = 0.05
DEPTH_LOSS_NAME = 'l1'
BATCH_SIZE = 32
NUM_EPOCHS = 100
TEST_BATCH_SIZE = 250

# --- Optimizer and scheduler settings ---
LR = 3e-4
LEARNING_RATE_NAME = 'manual'
LEARNING_RATE_OPTIONS = {LEARNING_RATE_NAME: LR}
WEIGHT_DECAY = 1e-4
BACKBONE_LEARNING_RATE = None
BACKBONE_LR_SCALE = None
BACKBONE_WARMUP_EPOCHS = 2
USE_LR_SCHEDULER = True
SCHEDULER_NAME = 'plateau'
SCHEDULER_METRIC = 'val_ADE'
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 6
SCHEDULER_MIN_LR = 1e-5
EARLY_STOPPING_PATIENCE = 20
EARLY_STOPPING_MIN_DELTA = 1e-3

# --- Inference and output behavior ---
RELOAD_BEST_CHECKPOINT_FOR_INFERENCE = True

EXPERIMENT_NAME = f"{MODEL_NAME}_{'depth' if USE_DEPTH_AUX else 'traj'}"
if USE_LR_SCHEDULER:
    EXPERIMENT_NAME += f'_{SCHEDULER_NAME}'
if WEIGHT_DECAY > 0:
    EXPERIMENT_NAME += f'_wd{WEIGHT_DECAY:g}'


## Environment Setup

The next cell bootstraps the repository in both local execution and Google Colab, then resolves the phase-aware output paths used throughout the notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def _is_running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def _find_project_root(start: Path) -> Path | None:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    return None


def _bootstrap_project_root() -> tuple[Path, bool]:
    in_colab = _is_running_in_colab()
    project_root = _find_project_root(Path.cwd())

    if in_colab:
        if project_root is None:
            git_dir = COLAB_PROJECT_DIR / '.git'
            if git_dir.is_dir():
                print(f'Updating repository in {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
                project_root = COLAB_PROJECT_DIR
            elif COLAB_PROJECT_DIR.exists():
                if (COLAB_PROJECT_DIR / 'src').is_dir() and (COLAB_PROJECT_DIR / 'notebooks').is_dir():
                    print(f'Using existing project directory in {COLAB_PROJECT_DIR}...')
                    project_root = COLAB_PROJECT_DIR
                else:
                    raise FileExistsError(f'{COLAB_PROJECT_DIR} exists but is not a recognized project root.')
            else:
                print(f'Cloning repository into {COLAB_PROJECT_DIR}...')
                subprocess.check_call(['git', 'clone', REPO_URL, str(COLAB_PROJECT_DIR)])
                project_root = COLAB_PROJECT_DIR
        elif project_root == COLAB_PROJECT_DIR and (COLAB_PROJECT_DIR / '.git').is_dir():
            print(f'Updating repository in {COLAB_PROJECT_DIR}...')
            subprocess.check_call(['git', '-C', str(COLAB_PROJECT_DIR), 'pull', '--ff-only'])
    elif project_root is None:
        raise FileNotFoundError('Could not find the project root. Open the notebook from inside the repository.')

    os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    return project_root.resolve(), in_colab


PROJECT_ROOT, IN_COLAB = _bootstrap_project_root()

from src.shared.project_setup import prepare_project_context

PROJECT = prepare_project_context(
    project_root=PROJECT_ROOT,
    in_colab=IN_COLAB,
    mount_drive_in_colab=MOUNT_DRIVE_IN_COLAB,
    drive_runs_root=COLAB_DRIVE_RUNS_ROOT,
    phase_name=PHASE_NAME,
)

PROJECT_ROOT = PROJECT.project_root
DATA_DIR = PROJECT.data_dir
TRAIN_DIR = PROJECT.train_dir
VAL_DIR = PROJECT.val_dir
TEST_DIR = PROJECT.test_dir
RUNS_DIR = PROJECT.runs_dir
CHECKPOINT_DIR = PROJECT.checkpoints_dir
SUBMISSION_DIR = PROJECT.submissions_dir
LEGACY_CHECKPOINT_PATH = PROJECT.legacy_checkpoint_path
LEGACY_SUBMISSION_PATH = PROJECT.legacy_submission_path

print(f'Environment: {"Google Colab" if PROJECT.in_colab else "Local"}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Data directory: {DATA_DIR}')
print(f'Run directory root: {RUNS_DIR}')
print(f'Checkpoint directory: {CHECKPOINT_DIR}')
print(f'Submission directory: {SUBMISSION_DIR}')


## Imports, Data Preparation, and Runtime Setup

The notebook now only orchestrates the Phase 2 flow. Dataset logic, model definitions, training, and submission helpers all live in `src/phase2/`.


In [ ]:
import os

import torch
from torch.utils.data import DataLoader

from src.phase2.dataset import DrivingDataset
from src.phase2.model import build_model
from src.phase2.submission import generate_submission
from src.phase2.train import train, validate
from src.shared.data_utils import DATASET_SPECS, ensure_all_datasets, has_pkl_files, sorted_pkl_files
from src.shared.logger import Logger
from src.shared.run_utils import (
    build_initial_run_metrics,
    copy_artifact_to_destination,
    create_run_context,
    save_metrics,
    sync_run_to_drive,
    write_summary,
)
from src.shared.training_setup import build_optimizer, build_scheduler

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 2 if (PROJECT.in_colab or os.name != 'nt') else 0
PIN_MEMORY = DEVICE.type == 'cuda'

if DOWNLOAD_DATA_IF_MISSING:
    ensure_all_datasets(data_dir=DATA_DIR, in_colab=PROJECT.in_colab, project_root=PROJECT_ROOT)
else:
    missing_splits = [
        name
        for name, config in DATASET_SPECS.items()
        if not has_pkl_files(DATA_DIR / config['target_subdir'])
    ]
    if missing_splits:
        missing_str = ', '.join(missing_splits)
        raise FileNotFoundError(
            f'Missing extracted dataset folders for: {missing_str}. '
            'Either enable DOWNLOAD_DATA_IF_MISSING or place the files manually under data/.'
        )

print(f'Device: {DEVICE}')
print(f'num_workers: {NUM_WORKERS}')
print(f'Experiment name: {EXPERIMENT_NAME}')
print(f'Model: {MODEL_NAME}')
print(f'Use depth auxiliary task: {USE_DEPTH_AUX}')
print(f'Lambda depth: {LAMBDA_DEPTH}')
print(f'LR: {LR} | Weight decay: {WEIGHT_DECAY}')


## Run Tracking

Run folders, logs, summaries, and submission paths are managed with the same shared utilities used by Phase 1.


In [ ]:
RUN_CONTEXT = create_run_context(
    project_root=PROJECT_ROOT,
    in_colab=PROJECT.in_colab,
    run_name=RUN_NAME,
    default_run_name=EXPERIMENT_NAME,
    drive_root=PROJECT.drive_runs_root,
    phase_name=PHASE_NAME,
)

RUN_METRICS = build_initial_run_metrics(
    RUN_CONTEXT,
    model_name=MODEL_NAME,
    device=str(DEVICE),
    batch_size=BATCH_SIZE,
    learning_rate_name=LEARNING_RATE_NAME,
    learning_rate_options=LEARNING_RATE_OPTIONS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    scheduler_enabled=USE_LR_SCHEDULER,
    scheduler_name=SCHEDULER_NAME,
    scheduler_metric=SCHEDULER_METRIC,
    num_epochs=NUM_EPOCHS,
    legacy_checkpoint_path=LEGACY_CHECKPOINT_PATH,
    legacy_submission_path=LEGACY_SUBMISSION_PATH,
)
RUN_METRICS.update(
    {
        'use_depth_aux': USE_DEPTH_AUX,
        'lambda_depth': LAMBDA_DEPTH,
        'depth_loss_name': DEPTH_LOSS_NAME,
    }
)

save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Run name: {RUN_CONTEXT.run_name}')
print(f'Run directory: {RUN_CONTEXT.run_dir}')
print(f'Best checkpoint path: {RUN_CONTEXT.checkpoint_path}')
print(f'Submission path: {RUN_CONTEXT.submission_path}')


## Datasets and Dataloaders

Phase 2 adds `driving_command` and optional depth supervision to the Phase 1 camera + history setup.


In [ ]:
train_files = sorted_pkl_files(TRAIN_DIR)
val_files = sorted_pkl_files(VAL_DIR)
test_files = sorted_pkl_files(TEST_DIR)

if not train_files:
    raise FileNotFoundError(f'No training files found in {TRAIN_DIR}')
if not val_files:
    raise FileNotFoundError(f'No validation files found in {VAL_DIR}')
if not test_files:
    raise FileNotFoundError(f'No test files found in {TEST_DIR}')

train_dataset = DrivingDataset(train_files, use_depth_aux=USE_DEPTH_AUX)
val_dataset = DrivingDataset(val_files, use_depth_aux=USE_DEPTH_AUX)
test_dataset = DrivingDataset(test_files, test=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

sample = train_dataset[0]
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)} | Test samples: {len(test_dataset)}')
print(f"Sample keys: {list(sample.keys())}")
print(f"camera: {tuple(sample['camera'].shape)} | history: {tuple(sample['history'].shape)} | driving_command: {sample['driving_command'].item()}")
print(f"future: {tuple(sample['future'].shape)}")
if USE_DEPTH_AUX:
    print(f"depth: {tuple(sample['depth'].shape)}")


## Model, Optimizer, Scheduler, and Training

Trajectory prediction remains the main task. When enabled, depth is used only as auxiliary supervision during training and validation.


In [ ]:
model = build_model(MODEL_NAME)
if USE_DEPTH_AUX and not getattr(model, 'supports_depth_aux', False):
    raise ValueError('USE_DEPTH_AUX=True requires a model with a depth head, such as phase2_multitask.')

optimizer = build_optimizer(
    model,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    backbone_learning_rate=BACKBONE_LEARNING_RATE,
    backbone_lr_scale=BACKBONE_LR_SCALE,
)
scheduler = build_scheduler(
    optimizer,
    enabled=USE_LR_SCHEDULER,
    name=SCHEDULER_NAME,
    factor=SCHEDULER_FACTOR,
    patience=SCHEDULER_PATIENCE,
    min_lr=SCHEDULER_MIN_LR,
)
logger = Logger(log_path=RUN_CONTEXT.log_path)

TRAINING_SUMMARY = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    logger,
    num_epochs=NUM_EPOCHS,
    scheduler=scheduler,
    scheduler_metric=SCHEDULER_METRIC,
    best_checkpoint_path=RUN_CONTEXT.checkpoint_path,
    lambda_depth=LAMBDA_DEPTH,
    use_depth_aux=USE_DEPTH_AUX,
    depth_loss_name=DEPTH_LOSS_NAME,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    backbone_warmup_epochs=BACKBONE_WARMUP_EPOCHS,
)

final_metrics = TRAINING_SUMMARY.get('final', {})
best_metrics = TRAINING_SUMMARY.get('best', {})
RUN_METRICS.update(
    {
        'epochs_completed': TRAINING_SUMMARY.get('epochs_completed'),
        'train_loss_final': final_metrics.get('train_total_loss'),
        'val_loss_final': final_metrics.get('val_total_loss'),
        'train_traj_loss_final': final_metrics.get('train_traj_loss'),
        'train_depth_loss_final': final_metrics.get('train_depth_loss'),
        'val_traj_loss_final': final_metrics.get('val_traj_loss'),
        'val_depth_loss_final': final_metrics.get('val_depth_loss'),
        'val_ADE_final': final_metrics.get('val_ADE'),
        'val_FDE_final': final_metrics.get('val_FDE'),
        'best_val_ADE': best_metrics.get('val_ADE'),
        'best_val_ADE_epoch': best_metrics.get('epoch'),
        'best_checkpoint_path': best_metrics.get('checkpoint_path'),
        'checkpoint_path': best_metrics.get('checkpoint_path'),
        'best_val_FDE_at_best_ADE': best_metrics.get('val_FDE'),
        'best_val_total_loss_at_best_ADE': best_metrics.get('val_total_loss'),
        'epoch_history': TRAINING_SUMMARY.get('history', []),
        'final_learning_rate': TRAINING_SUMMARY.get('final_learning_rate'),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

if best_metrics:
    print(f"Best val ADE: {best_metrics['val_ADE']:.4f} at epoch {best_metrics['epoch']}")
    print(f"Best checkpoint: {best_metrics.get('checkpoint_path')}")
print(f'Run metrics updated: {RUN_CONTEXT.metrics_path}')
print(f'Run log: {RUN_CONTEXT.log_path}')


## Reload Best Checkpoint and Validate

This step mirrors the Phase 1 workflow: save the last checkpoint, copy the best checkpoint into the legacy phase-scoped location, optionally reload the best checkpoint, and then compute a clean validation summary.


In [ ]:
LAST_CHECKPOINT_PATH = RUN_CONTEXT.run_dir / 'model_last.pth'
torch.save(model.state_dict(), LAST_CHECKPOINT_PATH)
copy_artifact_to_destination(RUN_CONTEXT.checkpoint_path, LEGACY_CHECKPOINT_PATH)

RUN_METRICS['last_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)
RUN_METRICS['best_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
RUN_METRICS['checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)

if RELOAD_BEST_CHECKPOINT_FOR_INFERENCE:
    best_state_dict = torch.load(RUN_CONTEXT.checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_state_dict)
    model = model.to(DEVICE)
    RUN_METRICS['inference_checkpoint_path'] = str(RUN_CONTEXT.checkpoint_path)
else:
    RUN_METRICS['inference_checkpoint_path'] = str(LAST_CHECKPOINT_PATH)

VALIDATION_METRICS = validate(
    model,
    val_loader,
    device=DEVICE,
    use_depth_aux=USE_DEPTH_AUX,
    lambda_depth=LAMBDA_DEPTH,
    depth_loss_name=DEPTH_LOSS_NAME,
)
RUN_METRICS.update(
    {
        'reloaded_val_ADE': VALIDATION_METRICS.get('val_ADE'),
        'reloaded_val_FDE': VALIDATION_METRICS.get('val_FDE'),
        'reloaded_val_traj_loss': VALIDATION_METRICS.get('val_traj_loss'),
        'reloaded_val_depth_loss': VALIDATION_METRICS.get('val_depth_loss'),
        'reloaded_val_total_loss': VALIDATION_METRICS.get('val_total_loss'),
    }
)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Best model saved to {RUN_CONTEXT.checkpoint_path}')
print(f'Last-epoch model saved to {LAST_CHECKPOINT_PATH}')
print(f'Historical checkpoint copy saved to {LEGACY_CHECKPOINT_PATH}')
print(VALIDATION_METRICS)


## Submission Generation

Submission uses only `camera`, `history`, and `driving_command`. Auxiliary depth supervision is never used as a test-time input or output.


In [ ]:
submission = generate_submission(
    model=model,
    data_loader=test_loader,
    device=DEVICE,
    output_path=RUN_CONTEXT.submission_path,
    legacy_output_path=LEGACY_SUBMISSION_PATH,
    copy_fn=copy_artifact_to_destination,
)

RUN_METRICS['submission_path'] = str(RUN_CONTEXT.submission_path)
RUN_METRICS['submission_shape'] = list(submission.shape)
save_metrics(RUN_CONTEXT, RUN_METRICS)
write_summary(RUN_CONTEXT, RUN_METRICS)

drive_backup_dir = sync_run_to_drive(RUN_CONTEXT) if SYNC_RUN_TO_DRIVE else None
if drive_backup_dir is not None:
    RUN_METRICS['drive_backup_enabled'] = True
    RUN_METRICS['drive_backup_path'] = str(drive_backup_dir)
    save_metrics(RUN_CONTEXT, RUN_METRICS)
    write_summary(RUN_CONTEXT, RUN_METRICS)

print(f'Submission saved to {RUN_CONTEXT.submission_path}')
print(f'Historical submission copy saved to {LEGACY_SUBMISSION_PATH}')
if drive_backup_dir is not None:
    print(f'Run backup synchronized to {drive_backup_dir}')
print(f'Submission shape: {submission.shape}')
